In [1]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine
import xgboost as xgb


In [2]:
# DB connection
engine = create_engine("postgresql+psycopg2://rirg2545@localhost/mimic")

In [3]:
# Mapping of drug ITEMIDs to labels and colors
ITEMID_LABELS = {
    221906: "Norepinephrine",
    221289: "Epinephrine",
    221662: "Dopamine",
    221986: "Milrinone",
    222315: "Vasopressin",
    221749: "Phenylephrine",
    221653: "Dobutamine",
    227692: "Isuprel"
}

ITEMID_COLORS = {
    221906: "blue",
    221289: "red",
    221662: "green",
    221986: "purple",
    222315: "orange",
    221749: "cyan",
    221653: "magenta",
    227692: "brown"
}

DATASETS = [
    "dobutamine", "dopamine", "epinephrine", "isuprel",
    "milrinone", "norepinephrine", "phenylephrine", "vasopressin"
]

# THRESHOLDS = {
#     221653: 0.000318,  # Dobutamine
#     221662: 0.001193,  # Dopamine
#     221289: 0.001011,  # Epinephrine
#     227692: 0.000004,  # Isuprel
#     221986: 0.000437,  # Milrinone
#     221906: 0.004640,  # Norepinephrine
#     221749: 0.006470,  # Phenylephrine
#     222315: 0.001269   # Vasopressin
#     mix: 
# }
THRESHOLDS = {
    "mix_80": 0.00115
}

MODELS_DIR = "../models/without_context_features"

In [4]:
# angepasst um treatment_count anzupassen
def preprocess_data(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Drop metadata/time columns and encode the label.
    """
    drop_cols = [
        "subject_id", "icustay_id",
        "context_start", "context_end",
        "target_start", "target_end", "treatment_given",
        "only_2_values"
    ]
    
    df_features = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # treatment_given already dropped ( treatment_count non-xistent)


    # Label as int
    df_features["label"] = df_features["positive_event"].astype(int)

    return df_features

def get_feature_cols(df_features: pd.DataFrame) -> list:
    """
    Identify feature columns (exclude label and split markers).
    """
    excluded = {"positive_event", "positive_sample", "split", "label"}
    feature_cols = [c for c in df_features.columns if c not in excluded]
    
    return feature_cols

def get_icustay_bounds(icustay_id):
    query = """
        SELECT intime, outtime
        FROM mimiciii.icustays
        WHERE icustay_id = %(icustay_id)s;
    """
    df = pd.read_sql(query, engine, params={"icustay_id": icustay_id})
    if df.empty:
        raise ValueError("ICU stay not found")
    return df.iloc[0]['intime'], df.iloc[0]['outtime']

def load_ground_truth(icustay_id):  
    
    df = pd.read_sql(f"""
        SELECT target_start, target_end
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s AND positive_event IS TRUE
    """, engine, params={"icustay_id": icustay_id})
    
    return df

def load_predictions(icustay_id):
    # 1) Load data (same table used for training)
    df = pd.read_sql(
        """
        SELECT *
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s
          AND split = 'test'
        ORDER BY context_end
        """,
        engine,
        params={"icustay_id": icustay_id},
    )

    if df.empty:
        return pd.DataFrame(
            columns=["target_start", "target_end", "pred_proba", "drug_label", "color"]
        )

    # 2) Datetime parsing (for plotting later)
    for col in ["context_start", "context_end", "target_start", "target_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # 3) Reproduce training preprocessing
    df_features = preprocess_data(
        df)

    feature_cols = get_feature_cols(df_features)

    X = df_features[feature_cols]
    y = df_features["label"]  # not strictly needed for plotting

    # 4) DMatrix (exactly like training)
    dmat = xgb.DMatrix(X, missing=np.nan)

    # 5) Load trained Booster
    model_path = "/dss/work/rirg2545/actionable-hypotension/models_given/optimized/xgb_mix.json"
    bst = xgb.Booster()
    bst.load_model(model_path)

    # 6) Predict
    preds = bst.predict(dmat)
    df = df.assign(pred_proba=preds)

    # 7) Thresholding (single model → single threshold)
    threshold = THRESHOLDS.get("mix_80", 0.01)
    pred_df = df[df["pred_proba"] >= threshold].copy()

    pred_df["drug_label"] = "Hypotension Risk"
    pred_df["color"] = "red"  # z.B. Signalrot für Warnung

    return pred_df[
        ["target_start", "target_end", "pred_proba", "drug_label", "color"]
    ]


def plot_blocks(fig, df, row):
    for entry in df.itertuples():
        fig.add_trace(go.Scatter(
            x=[entry.target_start, entry.target_end],
            y=[1, 1],
            mode="lines",
            line=dict(color="red", width=10),
            name="Target Window",
            showlegend=False,
            opacity=0.3,
            hovertemplate=f"Target Window<br>%{{x}}"
        ), row=row, col=1)
    fig.update_yaxes(title="Target Windows", row=row)

In [5]:
def load_available_context_windows(icustay_id):
    
    df = pd.read_sql(f"""
        SELECT context_start, context_end
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s
    """, engine, params={"icustay_id": icustay_id})

    return df

def plot_context_windows(fig, df, row):
    # Da keine itemid mehr existiert, alles in einer Zeile plotten (y=1)
    for entry in df.itertuples():
        fig.add_trace(go.Scatter(
            x=[entry.context_start, entry.context_end],
            y=[1, 1],  # Nur eine Zeile für alle Kontextfenster
            mode="lines",
            line=dict(color="gray", width=10),
            name="Context Window",
            showlegend=False,
            opacity=0.3,
            hovertemplate=f"Context Window<br>%{{x}}"
        ), row=row, col=1)

    fig.update_yaxes(title="Context Windows", row=row, range=[0.5, 1.5], showticklabels=False)

In [6]:
def load_treatments(icustay_id):
    query = """
        SELECT treatment_starttime, treatment_endtime, itemid
        FROM ce_approach.linkorder_treatment_events
        WHERE icustay_id = %(icustay_id)s;
    """
    df = pd.read_sql(query, engine, params={"icustay_id": icustay_id})
    df["drug_label"] = df["itemid"].map(ITEMID_LABELS).fillna(df["itemid"].astype(str))
    df["color"] = df["itemid"].map(ITEMID_COLORS).fillna("gray")
    return df

def plot_treatments(fig, df, row):
    row_offset_map = {itemid: i for i, itemid in enumerate(sorted(df["itemid"].unique()))}
    for entry in df.itertuples():
        y = row_offset_map[entry.itemid] + 1
        fig.add_trace(go.Scatter(
            x=[entry.treatment_starttime, entry.treatment_endtime],
            y=[y, y],
            mode="lines",
            line=dict(color=entry.color, width=10),
            name=entry.drug_label,
            showlegend=False,
            hovertemplate=f"{entry.drug_label}<br>%{{x}}"
        ), row=row, col=1)
    fig.update_yaxes(title="Treatment", row=row)

In [7]:
def load_map_from_mix_windows(icustay_id):
    df = pd.read_sql(
        """
        SELECT
            context_start,
            context_end,
            map_values_filtered
        FROM ce_approach.mix_windows
        WHERE icustay_id = %(icustay_id)s
        ORDER BY context_start
        """,
        engine,
        params={"icustay_id": icustay_id},
    )

    rows = []
    for _, r in df.iterrows():
        for e in r.map_values_filtered:
            charttime = r.context_start + pd.to_timedelta(e["pos"], unit="s")
            valuenum = e["value"]
            rows.append(
                {
                    "charttime": charttime,
                    "valuenum": valuenum
                }
            )

    return pd.DataFrame(rows)

def plot_map_values(fig, df, row):
    fig.add_trace(go.Scatter(
        x=df["charttime"],
        y=df["valuenum"],
        mode="markers+lines",
        line=dict(width=1),
        marker=dict(size=4),
        name="MAP (mmHg)",
        marker_color="black",
        showlegend=False,
        hovertemplate="MAP @ %{x}: %{y} mmHg"
    ), row=row, col=1)
    fig.update_yaxes(title="MAP (mmHg)", row=row)

In [8]:
def plot_positive_and_predicted_windows(icustay_id):
    treat = load_treatments(icustay_id) # treatment_events that actually happened
    ctx = load_available_context_windows(icustay_id) #context_start, context_end
    map_df = load_map_from_mix_windows(icustay_id) # load map values the model saw
    pos = load_ground_truth(icustay_id)
    pred = load_predictions(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id) # get intime outtime

    fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=(
                                        "Drug Treatments",
                                        "Available Context Windows",
                                        "MAP Values",
                                        "Positive Events (Ground Truth)", 
                                        "Model Predictions" 
                                        ))

    plot_treatments(fig, treat, row=1)
    plot_context_windows(fig, ctx, row=2)
    plot_map_values(fig, map_df, row=3)
    plot_blocks(fig, pos, row=4)
    plot_blocks(fig, pred, row=5)

    fig.update_layout(
        height=1000,
        title=f"Treatment Windows for ICU Stay {icustay_id}",
        xaxis=dict(range=[intime, outtime]),
        xaxis2=dict(range=[intime, outtime]),
        xaxis3=dict(range=[intime, outtime]),
        xaxis4=dict(range=[intime, outtime]),
        xaxis5=dict(range=[intime, outtime])
    )
    
    #fig.write_html(f"../plots/{icustay_id}.html")
    fig.show()

In [9]:
plot_positive_and_predicted_windows(icustay_id=200349)

In [10]:
plot_positive_and_predicted_windows(icustay_id=200349)

In [17]:
## GIF production
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
from datetime import timedelta
import imageio.v2 as imageio
from pathlib import Path
import tempfile

def create_prediction_gif(icustay_id, interval_minutes=15, frames_around_event=10, output_path=None):
    """
    Create a GIF showing how model predictions evolve over time, focused on ground truth events.
    
    Parameters:
    -----------
    icustay_id : int
        The ICU stay identifier
    interval_minutes : int
        Time interval between frames in minutes (default: 15)
    frames_around_event : int
        Number of frames to show before and after each ground truth event (default: 10)
    output_path : str or None
        Path to save the GIF. If None, saves to f"../plots/{icustay_id}_predictions.gif"
    """
    # Load all data
    treat = load_treatments(icustay_id)
    map_df = load_map_from_mix_windows(icustay_id)
    pos = load_ground_truth(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id)
    
    # Load all predictions data
    all_predictions_df = load_all_predictions_data(icustay_id)
    
    if all_predictions_df.empty:
        print(f"No test data found for ICU stay {icustay_id}")
        return
    
    if pos.empty:
        print(f"No ground truth events found for ICU stay {icustay_id}")
        return
    
    # Generate time points focused on ground truth events
    time_points = generate_focused_time_points(
        pos, interval_minutes, frames_around_event, intime, outtime
    )
    
    print(f"Generated {len(time_points)} frames focused on {len(pos)} ground truth events")
    print(f"Time range: {time_points[0]} to {time_points[-1]}")
    
    # Create temporary directory for frames
    temp_dir = tempfile.mkdtemp()
    frame_paths = []
    
    print(f"Generating frames...")
    
    for i, current_time in enumerate(time_points):
        if i % 10 == 0:
            print(f"Frame {i+1}/{len(time_points)}: {current_time}")
        
        # Filter data up to current time
        pred_at_time = filter_predictions_by_time(all_predictions_df, current_time)
        map_at_time = map_df[map_df['charttime'] <= current_time]
        
        # Create plot for this time point
        frame_path = Path(temp_dir) / f"frame_{i:04d}.png"
        create_timeline_frame(
            icustay_id, treat, map_at_time, pos, pred_at_time,
            intime, outtime, current_time, frame_path
        )
        frame_paths.append(str(frame_path))
    
    # Create GIF
    if output_path is None:
        output_path = f"/dss/work/rirg2545/actionable-hypotension/simulation/{icustay_id}_predictions.gif"
    
    print(f"Creating GIF at {output_path}...")
    create_gif_from_frames(frame_paths, output_path, duration=0.5)
    
    # Cleanup
    for frame_path in frame_paths:
        Path(frame_path).unlink()
    Path(temp_dir).rmdir()
    
    print(f"GIF created successfully: {output_path}")

def create_prediction_video(icustay_id, interval_minutes=15, frames_around_event=10, output_path=None, fps=2):
    """Create an MP4 video instead of GIF"""
    # Load all data
    treat = load_treatments(icustay_id)
    map_df = load_map_from_mix_windows(icustay_id)
    pos = load_ground_truth(icustay_id)
    intime, outtime = get_icustay_bounds(icustay_id)
    
    # Load all predictions data
    all_predictions_df = load_all_predictions_data(icustay_id)
    
    if all_predictions_df.empty:
        print(f"No test data found for ICU stay {icustay_id}")
        return
    
    if pos.empty:
        print(f"No ground truth events found for ICU stay {icustay_id}")
        return
    
    # Generate time points focused on ground truth events
    time_points = generate_focused_time_points(
        pos, interval_minutes, frames_around_event, intime, outtime
    )
    
    print(f"Generated {len(time_points)} frames focused on {len(pos)} ground truth events")
    print(f"Time range: {time_points[0]} to {time_points[-1]}")
    
    # Create temporary directory for frames
    temp_dir = tempfile.mkdtemp()
    frame_paths = []
    
    print(f"Generating frames...")
    
    for i, current_time in enumerate(time_points):
        if i % 10 == 0:
            print(f"Frame {i+1}/{len(time_points)}: {current_time}")
        
        # Filter data up to current time
        pred_at_time = filter_predictions_by_time(all_predictions_df, current_time)
        map_at_time = map_df[map_df['charttime'] <= current_time]
        
        # Create plot for this time point
        frame_path = Path(temp_dir) / f"frame_{i:04d}.png"
        create_timeline_frame(
            icustay_id, treat, map_at_time, pos, pred_at_time,
            intime, outtime, current_time, frame_path
        )
        frame_paths.append(str(frame_path))
    if output_path is None:
        output_path = f"/dss/work/rirg2545/actionable-hypotension/simulation/{icustay_id}_predictions.mp4"
    
    # Create video with imageio
    writer = imageio.get_writer(output_path, fps=fps, codec='libx264', quality=8)
    
    for frame_path in frame_paths:
        frame = imageio.imread(frame_path)
        writer.append_data(frame)
    
    writer.close()


def create_timeline_frame(icustay_id, treat, map_df, pos, pred, 
                          intime, outtime, current_time, save_path):
    """
    Create a simple timeline visualization showing:
    1. MAP values over time (context)
    2. Ground truth positive events
    3. Model predictions
    4. Treatment starts
    """
    fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
    fig.suptitle(f'ICU Stay {icustay_id} - Time: {current_time.strftime("%Y-%m-%d %H:%M")}', 
                 fontsize=16, fontweight='bold')
    
    # Convert times to hours from admission for easier plotting
    def to_hours(dt):
        return (dt - intime).total_seconds() / 3600
    
    current_hour = to_hours(current_time)
    total_hours = to_hours(outtime)
    
    # ============== Panel 1: MAP Values ==============
    ax1 = axes[0]
    if not map_df.empty:
        hours = [to_hours(t) for t in map_df['charttime']]
        ax1.plot(hours, map_df['valuenum'], 'k-', linewidth=1, marker='o', markersize=3)
        ax1.axhline(y=65, color='red', linestyle='--', alpha=0.3, label='Hypotension Threshold (65mmHg)')
    ax1.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='Current time')
    ax1.set_ylabel('MAP (mmHg)', fontsize=12, fontweight='bold')
    ax1.set_title('Context: Mean Arterial Pressure', fontsize=11, loc='left')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right')
    ax1.set_ylim(40, 120)
    
    # ============== Panel 2: Ground Truth Positive Events ==============
    ax2 = axes[1]
    ax2.set_ylabel('Ground Truth', fontsize=12, fontweight='bold')
    ax2.set_title('True Catecholamine Initiation', fontsize=11, loc='left')
    ax2.set_ylim(0, 2)
    ax2.set_yticks([])
    
    for _, event in pos.iterrows():
        start_hour = to_hours(event['target_start'])
        end_hour = to_hours(event['target_end'])
        width = end_hour - start_hour
        
        # Only show if event has started by current time
        if start_hour <= current_hour:
            # Color based on whether event is in past or ongoing
            if end_hour <= current_hour:
                color = 'darkred'
                alpha = 0.5
            else:
                color = 'red'
                alpha = 0.8
            
            rect = Rectangle((start_hour, 0.5), width, 1.0, 
                           facecolor=color, edgecolor='black', alpha=alpha, linewidth=1)
            ax2.add_patch(rect)
    
    ax2.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax2.grid(True, alpha=0.3, axis='x')
    
    # ============== Panel 3: Model Predictions ==============
    ax3 = axes[2]
    ax3.set_ylabel('Predictions', fontsize=12, fontweight='bold')
    ax3.set_title('Model Predicted Risk Windows', fontsize=11, loc='left')
    ax3.set_ylim(0, 2)
    ax3.set_yticks([])
    
    if not pred.empty:
        for _, p in pred.iterrows():
            start_hour = to_hours(p['target_start'])
            end_hour = to_hours(p['target_end'])
            width = end_hour - start_hour
            
            # Predictions fade based on when they were made
            pred_time = to_hours(p['context_end'])
            hours_old = current_hour - pred_time
            alpha = max(0.3, 1.0 - (hours_old / 24))  # Fade over 24 hours
            
            rect = Rectangle((start_hour, 0.5), width, 1.0,
                           facecolor='orange', edgecolor='black', alpha=alpha, linewidth=1)
            ax3.add_patch(rect)
    
    ax3.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax3.grid(True, alpha=0.3, axis='x')
    
    # ============== Panel 4: Treatment Starts ==============
    ax4 = axes[3]
    ax4.set_ylabel('Treatments', fontsize=12, fontweight='bold')
    ax4.set_title('Vasopressor Treatment Initiations', fontsize=11, loc='left')
    ax4.set_ylim(0, 2)
    ax4.set_yticks([])
    ax4.set_xlabel('Hours from ICU Admission', fontsize=12, fontweight='bold')
    
    # Show treatment starts as vertical lines
    treat_at_time = treat[treat['treatment_starttime'] <= current_time]
    for _, t in treat_at_time.iterrows():
        start_hour = to_hours(t['treatment_starttime'])
        ax4.axvline(x=start_hour, color='green', linewidth=3, alpha=0.7)
        ax4.text(start_hour, 1.5, t['drug_label'], rotation=90, 
                va='bottom', ha='right', fontsize=8)
    
    ax4.axvline(x=current_hour, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax4.grid(True, alpha=0.3, axis='x')
    
    # Set x-axis limits for all panels
    for ax in axes:
        ax.set_xlim(0, total_hours)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close()

def generate_focused_time_points(pos_df, interval_minutes, frames_around, intime, outtime):
    """
    Generate time points focused around ground truth events.
    """
    time_points_set = set()
    interval_delta = timedelta(minutes=interval_minutes)
    
    for _, event in pos_df.iterrows():
        # Find center of ground truth event
        event_center = event['target_start'] + (event['target_end'] - event['target_start']) / 2
        
        # Generate frames around this event
        for offset in range(-frames_around, frames_around + 1):
            time_point = event_center + (offset * interval_delta)
            
            # Keep within ICU stay bounds
            if intime <= time_point <= outtime:
                time_points_set.add(time_point)
    
    # Sort time points
    time_points = sorted(list(time_points_set))
    
    return time_points

def load_all_predictions_data(icustay_id):
    """Load all prediction data without filtering by time."""
    df = pd.read_sql(
        """
        SELECT *
        FROM ce_approach.merged_mix_features_updated
        WHERE icustay_id = %(icustay_id)s
          AND split = 'test'
        ORDER BY context_end
        """,
        engine,
        params={"icustay_id": icustay_id},
    )
    
    if df.empty:
        return pd.DataFrame()
    
    # Parse datetime columns
    for col in ["context_start", "context_end", "target_start", "target_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    # Preprocess and predict
    df_features = preprocess_data(df)
    feature_cols = get_feature_cols(df_features)
    X = df_features[feature_cols]
    
    dmat = xgb.DMatrix(X, missing=np.nan)
    
    # Load model
    model_path = "/dss/work/rirg2545/actionable-hypotension/models_given/optimized/xgb_mix.json"
    bst = xgb.Booster()
    bst.load_model(model_path)
    
    # Predict
    preds = bst.predict(dmat)
    df = df.assign(pred_proba=preds)
    
    # Apply threshold
    threshold = THRESHOLDS.get("mix_80", 0.01)
    df["predicted_positive"] = df["pred_proba"] >= threshold
    
    return df

def filter_predictions_by_time(df, current_time):
    """Filter predictions to only show those available at current_time."""
    if df.empty:
        return pd.DataFrame(
            columns=["target_start", "target_end", "pred_proba", "context_end"]
        )
    
    # Only show predictions where the context_end is <= current_time
    pred_df = df[
        (df["context_end"] <= current_time) & 
        (df["predicted_positive"] == True)
    ].copy()
    
    return pred_df[
        ["target_start", "target_end", "pred_proba", "context_end"]
    ]

def create_gif_from_frames(frame_paths, output_path, duration=0.5):
    """Create a GIF from a list of image paths."""
    images = []
    for path in frame_paths:
        images.append(imageio.imread(path))
    
    imageio.mimsave(output_path, images, duration=duration, loop=0)





In [18]:
#create_prediction_gif(icustay_id=220527, interval_minutes=15)
create_prediction_video(icustay_id=229240, interval_minutes=15)


Generated 126 frames focused on 6 ground truth events
Time range: 2106-01-14 08:56:32 to 2106-01-19 07:56:32
Generating frames...
Frame 1/126: 2106-01-14 08:56:32
Frame 11/126: 2106-01-14 11:26:32
Frame 21/126: 2106-01-14 13:56:32
Frame 31/126: 2106-01-14 20:56:32
Frame 41/126: 2106-01-14 23:26:32
Frame 51/126: 2106-01-15 07:41:32
Frame 61/126: 2106-01-15 10:11:32
Frame 71/126: 2106-01-17 16:41:32
Frame 81/126: 2106-01-17 19:11:32
Frame 91/126: 2106-01-18 09:41:32
Frame 101/126: 2106-01-18 12:11:32
Frame 111/126: 2106-01-19 04:11:32
Frame 121/126: 2106-01-19 06:41:32


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1589, 985) to (1600, 992) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
[rawvideo @ 0x6881080] Stream #0: not enough frames to estimate rate; consider increasing probesize
